In [1]:
import numpy as np
import pandas as pd
import torch
import json
from collections import defaultdict
from sklearn.preprocessing import StandardScaler

In [2]:
import json
with open(r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\vocab_dict_data\categorical_maps.json", "rb") as file:
    categorical_maps = json.load(file)
categorical_maps

{'department': {'Agricultural Engineering': 1,
  'Agricultural Sciences': 2,
  'Agricultural and Biological Engineering': 3,
  'Agricultural and Environmental Sciences': 4,
  'Agronomy': 5,
  'Amity Institute of Applied Sciences (Noida)': 6,
  'Amity School of Physical Sciences': 7,
  'Amrut Mody School of Management': 8,
  'Biochemistry': 9,
  'Bioinformatics': 10,
  'Bioinformatics and Computational Biology': 11,
  'Biological Engineering': 12,
  'Biological Sciences': 13,
  'Chemical Engineering': 14,
  'Chemical and Biochemical Processing Division': 15,
  'Chemistry': 16,
  'Civil and Environmental Engineering': 17,
  'Computational Biology': 18,
  'Computer Science': 19,
  'Computer Science and Artificial Intelligence Laboratory': 20,
  'Computer Science and Artificial Intelligence Laboratory (CSAIL)': 21,
  'Computer Science and Engineering': 22,
  'Computer Science and Media Arts': 23,
  'Computer Science and Media Lab': 24,
  'Computer Science and Technology': 25,
  'Department

In [3]:
PROF_prefinal_dataset = defaultdict()

In [4]:
df = pd.read_csv(r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\prof\extracted_04_prof_data.csv")
df.head()

,id,name,expertise,department,title,topics_display_name,concepts_display_name,primary_topics_display_name,abstract
0,1,Jaya Madan,Electrical and Electronic Engineering,Department of Electronics and Communication En...,Optimizing Tin-based Solar Cells: Unleashing t...,"{'Physical Sciences', 'Materials Science', 'En...","{'Materials science', 'Organic chemistry', 'Re...","{'Materials science', 'Organic chemistry', 'Re...",A sustainable and environmentally conscious fu...
1,2,Jaya Madan,Electrical and Electronic Engineering,Department of Electronics and Communication En...,Enhancing Photovoltaic Performance of Lead-Fre...,"{'Physical Sciences', 'Materials Chemistry', '...","{'Engineering physics', 'Optoelectronics', 'Ma...","{'Engineering physics', 'Optoelectronics', 'Ma...",Tin-based photovoltaic (PV) cells have become ...
2,3,Jaya Madan,Electrical and Electronic Engineering,Department of Electronics and Communication En...,Enhancing the Performance of CsSnCl<sub>3</sub...,"{'Physical Sciences', 'Materials Chemistry', '...","{'Optoelectronics', 'Materials science', 'Pero...","{'Optoelectronics', 'Materials science', 'Pero...","Recently, there has been a lot of interest in ..."
3,4,Jaya Madan,Electrical and Electronic Engineering,Department of Electronics and Communication En...,Maximizing Photovoltaic Efficiency: Thickness ...,"{'Physical Sciences', 'Materials Chemistry', '...",{'Computer science'},{'Computer science'},The Dion-Jacobson (DJ) structure developed lot...
4,5,Jaya Madan,Electrical and Electronic Engineering,Department of Electronics and Communication En...,Exploring Photovoltaic Efficiency in Perovskit...,"{'Physical Sciences', 'Perovskite Materials an...","{'Composite material', 'Engineering physics', ...","{'Composite material', 'Engineering physics', ...",Cesium tin chloride (CsSnCl3) exhibits promisi...


In [14]:
text_col = ["title", "abstract", "topics_display_name", "concepts_display_name", "primary_topics_display_name", "expertise"]
categorical_col = ["department"]
numerical_col = [""]

In [10]:
import sys
import os

# Use __file__ if running as a script, fallback to os.getcwd() in Jupyter notebook
try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    notebook_dir = os.getcwd()

# Go up two levels from 'src/utils' to reach the project root directory
project_root = os.path.abspath(os.path.join(notebook_dir, "..", ".."))

if project_root not in sys.path:
    sys.path.append(project_root)

from src.model_utils.candidate_tower_utils.categorical_encoder import CandidateCategoricalEncoder
from src.model_utils.candidate_tower_utils.numerical_encoder import CandidateNumercalEncoder
# from src.model_utils.query_tower_utils.numerical_encoder import QueryNumericalEncoder


In [17]:
import pandas as pd
import torch


def build_prof_categorical_dataset(
    df,
    categorical_cols,
    categorical_maps,
    categorical_encoder,
    id_col="id",
    device=None,
):
    """Maps the professor dataframe's categorical columns to IDs, runs them

    through the CandidateCategoricalEncoder, and returns
    PROF_prefinal_dataset.

    Automatically handles missing columns (like 'university' or 'country')
    by populating them with zeros (index for 'UNKNOWN') to match the expected
    input structure of the encoder.
    """
    if device is None:
        device = next(categorical_encoder.parameters()).device

    num_rows = len(df)

    # 1. Map columns present in the dataset to their IDs
    mapped_df = pd.DataFrame(index=df.index)
    for col in categorical_cols:
        col_map = categorical_maps.get(col, {})
        mapped_df[col] = (
            df[col]
            .fillna("UNKNOWN")
            .astype(str)
            .map(lambda x: col_map.get(x, col_map.get("UNKNOWN", 0)))
        )

    # Convert mapped values to a PyTorch tensor
    cat_ids_tensor = torch.tensor(
        mapped_df[categorical_cols].values, dtype=torch.long, device=device
    )

    # 2. Build the input dictionary expected by CandidateCategoricalEncoder
    categorical_inputs = {}
    for idx, col in enumerate(categorical_cols):
        categorical_inputs[col] = cat_ids_tensor[:, idx]

    # Fill any missing columns expected by the encoder with zeros (UNKNOWN index)
    for col in categorical_encoder.embeddings.keys():
        if col not in categorical_inputs:
            categorical_inputs[col] = torch.zeros(
                num_rows, dtype=torch.long, device=device
            )

    # 3. Generate embeddings with gradients disabled
    categorical_encoder.eval()
    with torch.no_grad():
        # Will output a 96-dimensional tensor (3 columns * 32 dims)
        cate_embeddings = categorical_encoder(categorical_inputs)

    # 4. Construct the final PROF_prefinal_dataset
    PROF_prefinal_dataset = []
    ids = df[id_col].values

    for idx, raw_id in enumerate(ids):
        # Store only the mapped IDs present in categorical_cols for 'cat_ids'
        PROF_prefinal_dataset.append(
            {
                "prof_id": int(raw_id),
                "cat_ids": cat_ids_tensor[idx],
                "cate_emb": cate_embeddings[idx],
            }
        )

    return PROF_prefinal_dataset


In [12]:
device = torch.device(
    "cuda" if torch.cuda.is_available()  else "cpu"
)

In [18]:
# from model_utils.candidate_tower_utils.categorical_encoder import CandidateCategoricalEncoder

# 1. Ensure categorical_maps exists for 'department'
# (If you don't already have one, here is how to build it dynamically from prof_df)
if "department" not in categorical_maps:
    unique_depts = sorted(df["department"].fillna("UNKNOWN").astype(str).unique())
    dept_map = {val: idx + 1 for idx, val in enumerate(unique_depts)}
    dept_map["UNKNOWN"] = 0
    categorical_maps["department"] = dept_map

# 2. Instantiate CandidateCategoricalEncoder and move it to the device
categorical_encoder = CandidateCategoricalEncoder(categorical_maps).to(device)

# 3. Define professor categorical columns (just "department")
categorical_col = ["department"]

# 4. Generate the prefinal dataset
PROF_prefinal_dataset = build_prof_categorical_dataset(
    df=df,
    categorical_cols=categorical_col,
    categorical_maps=categorical_maps,
    categorical_encoder=categorical_encoder,
    id_col="id", # The ID column we added to your prof CSV
    device=device
)

# 5. Verify the result
print("First row of the professor dataset:")
print(PROF_prefinal_dataset[0].keys())
print("prof_id:", PROF_prefinal_dataset[0]["prof_id"])
print("cat_ids:", PROF_prefinal_dataset[0]["cat_ids"])
print("cate_emb shape:", PROF_prefinal_dataset[0]["cate_emb"].shape)


First row of the professor dataset:
dict_keys(['prof_id', 'cat_ids', 'cate_emb'])
prof_id: 1
cat_ids: tensor([60])
cate_emb shape: torch.Size([160])


In [19]:
PROF_prefinal_dataset[0]

{'prof_id': 1,
 'cat_ids': tensor([60]),
 'cate_emb': tensor([-0.0835,  0.2471,  0.2513, -0.1258,  0.1065, -0.2173, -0.2118, -0.2082,
         -0.0890, -0.1844, -0.2036, -0.3537,  0.2065, -0.1148, -0.0032, -0.2507,
          0.1750,  0.0485,  0.3624,  0.1086, -0.1009,  0.2338, -0.1166, -0.1093,
          0.1599, -0.1306, -0.0094,  0.1060, -0.0250,  0.0972, -0.0573, -0.2128,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0

In [20]:
import ast
import pandas as pd


def parse_and_format_value(val):
    """Safely parses stringified Python collections (lists, sets, dicts)

    and formats them into comma-separated text.
    """
    if pd.isna(val):
        return ""

    # 1. If it's a string representation of a Python structure, parse it
    if isinstance(val, str):
        val_stripped = val.strip()
        if val_stripped.startswith(("{", "[", "(")) and val_stripped.endswith(
            ("}", "]", ")")
        ):
            try:
                val = ast.literal_eval(val_stripped)
            except Exception:
                pass  # Fallback to keeping it as a raw string

    # 2. Format the value based on its parsed type
    if isinstance(val, (list, set, tuple)):
        # Join list/set elements with commas
        items = [str(item).strip() for item in val if pd.notna(item)]
        return ", ".join(items)

    elif isinstance(val, dict):
        # Format dictionary as key-value pairs
        items = []
        for k, v in val.items():
            if isinstance(v, (list, set, tuple)):
                v_str = ", ".join([str(x) for x in v])
                items.append(f"{k}: {v_str}")
            else:
                items.append(f"{k}: {v}")
        return ", ".join(items)

    # 3. Default fallback for standard strings or numbers
    return str(val).strip()


In [21]:
def generate_prof_combined_texts(df, text_cols):
    """Combines specified text columns for each row into a single string,

    handling sets, dicts, and lists.
    """
    combined_texts = []

    for _, row in df.iterrows():
        row_pieces = []
        for col in text_cols:
            raw_val = row[col]
            val_str = parse_and_format_value(raw_val)

            # Format as "Column name: text value"
            if val_str:
                col_prefix = col.replace("_", " ").capitalize()
                row_pieces.append(f"{col_prefix}: {val_str}")

        # Combine all parts with a period and space
        row_text = ". ".join(row_pieces)
        if row_text:
            row_text += "."

        combined_texts.append(row_text)

    return combined_texts


In [22]:
# 1. Define the professor text columns
text_col = [
    "title",
    "abstract",
    "topics_display_name",
    "concepts_display_name",
    "primary_topics_display_name",
    "expertise",
]

# 2. Create the target DataFrame
prof_text_df = pd.DataFrame()

# 3. Combine the text
prof_text_df["texts"] = generate_prof_combined_texts(df, text_col)

# 4. View a sample of the first row's combined text
print("Sample text output:")
print(prof_text_df["texts"].iloc[0][:600])  # First 600 characters of row 0


Sample text output:
Title: Optimizing Tin-based Solar Cells: Unleashing the Potential of CsSnI<sub>3</sub> Perovskite for a Sustainable Energy Future. Abstract: A sustainable and environmentally conscious future is provided by renewable energy sources, notably solar energy, which have become a crucial element of the global energy revolution. Tin (Sn)-based solar cells are thought to be a competitive challenger in the PV (photovoltaic) market. PV devices based on cesium tin triiodide (CsSnI3) material are still trailing in performance, and they have not yet attained high conversion efficiency. The influence of thi


In [23]:
from sentence_transformers import SentenceTransformer
# 1. Initialize raw SentenceTransformer and move to device
text_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
text_encoder.to(device)

c:\Users\ps302\anaconda3\envs\genai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3253.04it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

In [24]:
import torch


def append_prof_text_embeddings(
    prefinal_dataset, texts, sentence_transformer, device=None
):
    """Encodes combined professor texts and appends them as 'text_emb' to each

    row in PROF_prefinal_dataset.
    """
    if len(prefinal_dataset) == 0:
        return prefinal_dataset

    if device is None:
        device = next(sentence_transformer.parameters()).device

    text_list = texts.tolist()

    # Move model to device
    sentence_transformer.to(device)

    # 1. Generate text embeddings in batch
    with torch.no_grad():
        all_embeddings = sentence_transformer.encode(
            text_list,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=True,
            device=device,
        )

    # 2. Append to dataset
    for idx, item in enumerate(prefinal_dataset):
        # Shape: (384,)
        item["text_emb"] = all_embeddings[idx]

    return prefinal_dataset


# Run the embedding process:
# (Uses the same text_encoder SentenceTransformer we defined earlier)
PROF_prefinal_dataset = append_prof_text_embeddings(
    prefinal_dataset=PROF_prefinal_dataset,
    texts=prof_text_df["texts"],
    sentence_transformer=text_encoder,
    device=device,
)

# 3. Verify the final keys
print("Updated Professor Dataset keys:")
print(PROF_prefinal_dataset[0].keys())
print("text_emb shape:", PROF_prefinal_dataset[0]["text_emb"].shape)


Batches: 100%|██████████| 292/292 [04:54<00:00,  1.01s/it]

Updated Professor Dataset keys:
dict_keys(['prof_id', 'cat_ids', 'cate_emb', 'text_emb'])
text_emb shape: torch.Size([384])


In [25]:
PROF_prefinal_dataset[0]

{'prof_id': 1,
 'cat_ids': tensor([60]),
 'cate_emb': tensor([-0.0835,  0.2471,  0.2513, -0.1258,  0.1065, -0.2173, -0.2118, -0.2082,
         -0.0890, -0.1844, -0.2036, -0.3537,  0.2065, -0.1148, -0.0032, -0.2507,
          0.1750,  0.0485,  0.3624,  0.1086, -0.1009,  0.2338, -0.1166, -0.1093,
          0.1599, -0.1306, -0.0094,  0.1060, -0.0250,  0.0972, -0.0573, -0.2128,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0

In [26]:
import json
import os
import pandas as pd
import torch

# 1. Define the target directory path and ensure it exists
target_dir = r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\prefinal_dataset\prof"
os.makedirs(target_dir, exist_ok=True)


# Helper function to recursively convert PyTorch tensors to standard Python lists
def convert_tensors_to_lists(obj):
    if isinstance(obj, dict):
        return {k: convert_tensors_to_lists(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_tensors_to_lists(item) for item in obj]
    elif isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()
    else:
        return obj


# Convert PyTorch tensors to lists for CSV and JSON serialization
clean_dataset = convert_tensors_to_lists(PROF_prefinal_dataset)





In [27]:
# =====================================================================
# OPTION 1: Save as CSV
# =====================================================================
csv_path = os.path.join(target_dir, "PROF_prefinal_dataset.csv")
df_csv = pd.DataFrame(clean_dataset)
df_csv.to_csv(csv_path, index=False)
print(f"Saved CSV successfully at:\n{csv_path}\n")

Saved CSV successfully at:
C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\prefinal_dataset\prof\PROF_prefinal_dataset.csv



In [28]:

# =====================================================================
# OPTION 2: Save as JSON
# =====================================================================
json_path = os.path.join(target_dir, "PROF_prefinal_dataset.json")
with open(json_path, "w") as file:
    json.dump(clean_dataset, file, indent=4)
print(f"Saved JSON successfully at:\n{json_path}\n")


Saved JSON successfully at:
C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\prefinal_dataset\prof\PROF_prefinal_dataset.json



In [29]:

# =====================================================================
# OPTION 3: Save directly as PyTorch Binary (keeps Tensors intact)
# =====================================================================
pt_path = os.path.join(target_dir, "PROF_prefinal_dataset.pt")
torch.save(PROF_prefinal_dataset, pt_path)
print(f"Saved PyTorch Binary successfully at:\n{pt_path}\n")

Saved PyTorch Binary successfully at:
C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\prefinal_dataset\prof\PROF_prefinal_dataset.pt



In [ ]:
# # Load the dataset back with tensors intact
# PHD_prefinal_dataset = torch.load(
#     r"C:\Users\ps302\OneDrive\Desktop\Recommend\src\data\processed\prefinal_dataset\PHD_prefinal_dataset.pt"
# )
